# Трек C: OWLv2 на соседних кадрах (dt = ±30)

Считает только OWLv2 и только кадры-соседи, которые сами не целевые: 879 кадров, 5173 кропа `top` ×4 (462 из них для train-строк).
Для целевых кадров детекции уже есть в `ov_owl__v1`, привязка берёт их по `src_frame_id`.

**Перед запуском**

1. Settings: Accelerator `GPU T4 x2`, Internet `Off`.
2. Add Input: датасет соревнования; датасет с кодом репозитория (коммит с флагом `--rows neighbours`); `throddy/ts-artifacts-ov-pseudo` (веса `models/ov_owl` и готовые файлы v1).
3. Save Version → Save & Run All.

**Выход** в `/kaggle/working/artifacts/dets/`: `ov_owl__v2.parquet` (доля 0/2, GPU 0) и `ov_owl__v3.parquet` (доля 1/2, GPU 1) с карточками, формат 8.1, `from_dt` равен −30 или 30.
Каждая доля останавливается штатно через `MAX_MINUTES`; при остановке подключить вывод как Input и запустить ещё раз, расчёт продолжится.

Оценка времени: в прошлом прогоне 4964 кропа OWLv2 заняли не больше 4.5 часа на одной T4 (точное время в журнале не записано), то есть на 2587 кропов на GPU нужно не больше 2.4 часа.
Тестовые кадры обрабатываются только автоматически.


In [ ]:
import os
from pathlib import Path

os.environ['TS_GIT_SHA'] = ''   # коммит репозитория, из которого собран архив с кодом
PROMPT_VERSION = 2              # тот же промпт, что у ov_owl__v1
MAX_MINUTES = 240               # штатная остановка каждой доли

os.environ['OV_PROMPT'] = str(PROMPT_VERSION)
os.environ['OV_MAX_MINUTES'] = str(MAX_MINUTES)
found = sorted(p.parents[2] for p in Path('/kaggle/input').rglob('tracks/ov/run_ov.py'))
assert len(found) == 1, f'Нужен ровно один подключённый датасет с кодом репозитория, найдено: {found}'
assert '--shard' in (found[0] / 'tracks/ov/run_ov.py').read_text(encoding='utf-8'), 'В датасете с кодом старая версия run_ov.py: нет --rows neighbours и --shard'
os.chdir(found[0])
%run tracks/ov/kaggle_setup.py


In [ ]:
%%bash
set -u
run() {  # доля, версия, GPU
  python tracks/ov/run_ov.py --source ov_owl --rows neighbours --shard "$1/2" --version "$2" \
    --prompt-version "$OV_PROMPT" --device "cuda:$3" --batch 2 --fp16 --max-minutes "$OV_MAX_MINUTES"
}
if [ "$(nvidia-smi -L | wc -l)" -ge 2 ]; then
  run 0 2 0 > /kaggle/working/owl_shard0.log 2>&1 &
  run 1 3 1 > /kaggle/working/owl_shard1.log 2>&1 &
  wait
else
  run 0 2 0 > /kaggle/working/owl_shard0.log 2>&1
  run 1 3 0 > /kaggle/working/owl_shard1.log 2>&1
fi
tail -n 5 /kaggle/working/owl_shard0.log /kaggle/working/owl_shard1.log


In [ ]:
art = Path(os.environ['TS_ARTIFACTS'])
for path in sorted((art / 'dets').glob('ov_owl__v[23].*')):
    print(f'{path.stat().st_size / 2 ** 20:8.2f} МБ  {path.name}')
!python -m ts.check dets {art}/dets/ov_owl__v2.parquet
!python -m ts.check dets {art}/dets/ov_owl__v3.parquet
